In [1]:
import os
import subprocess
import time
from src.functions import save_video_frames, copy_frame_to_folder
from PIL import ImageGrab, Image
import pygetwindow as gw
import numpy as np
from skimage.metrics import structural_similarity as ssim


In [ ]:
FRAMES_CLASSES = {
    'o': 'Data/Frames_Categories/Other',
    'tm': 'Data/Frames_Categories/Team_menu',
    's': 'Data/Frames_Categories/Salary_auction',
    'g': 'Data/Frames_Categories/Game',
    'cg': 'Data/Frames_Categories/Cup_game',
    'ge': 'Data/Frames_Categories/Game_extra_time',
    'gi': 'Data/Frames_Categories/Game_Interval',
    't': 'Data/Frames_Categories/Class_tables',
    'sq': 'Data/Frames_Categories/Squad',
    'cd': 'Data/Frames_Categories/Cup_draw',
    'ts': 'Data/Frames_Categories/Top_scorers',
    'mc': 'Data/Frames_Categories/Manager_changes',
    'pd': 'Data/Frames_Categories/Player_demand',
}

## Video

In [ ]:
save_video_frames('Data/Video/Elifoot_vid1.mp4', 'Data/Frames', 1) 
#Get frames from videoo

### Frame labeller code

In [ ]:
import os
from PIL import Image

frames_folder = 'Data/Frames'

for frame in os.listdir(frames_folder):
    frame_path = os.path.join(frames_folder, frame)
    img = Image.open(frame_path)
    img.resize((200, 200)).show(title = frame_path)
    class_choice = input('Frame_class')
    if class_choice == 'sk':
        continue
    copy_frame_to_folder(frame_path, class_choice)
    img.close()

## Simulation

In [ ]:
class EliBot:
    def __init__(self, game_path, dosbox_folder_path):

        self.game_path = game_path
        self.dosbox_folder_path = dosbox_folder_path

        """
        Launch game in DOSBox (DOS emulator)
        """
        # Start DOSBox with the game
        self.process = subprocess.Popen(
            self.game_path,
            cwd=self.dosbox_folder_path,
            stdin=subprocess.PIPE
        )
        time.sleep(3)  # Wait for game to load
        
        # Get game window position
        self.find_game_window()
    
    def find_game_window(self):
        """Find DOSBox window coordinates"""
        
        elifoot_windows = [title for title in gw.getAllTitles() if 'ELIFOOT2' in title]
        if len(elifoot_windows) > 1:
            raise ValueError('Multiple Elifoot windows found, exiting')
        if len(elifoot_windows) == 0:
            raise ValueError('No Elifoot window found, exiting')
        windows = gw.getWindowsWithTitle('DOSBox')
        self.game_window = windows[0]
        self.game_window.activate()
        
    def capture_screen(self):
        """Capture game screen"""
        if hasattr(self, 'game_window'):
            bbox = (
                self.game_window.left,
                self.game_window.top,
                self.game_window.right,
                self.game_window.bottom
            )
            screenshot = ImageGrab.grab(bbox)
            return np.array(screenshot)
        else:
            # Fallback: capture full screen
            screenshot = ImageGrab.grab()
            return np.array(screenshot)
    
    def analyze_screen(self, screen):
        """Analyze screen to extract game state"""
        # Convert to grayscale for analysis
        from PIL import Image
        img = Image.fromarray(screen)
        gray = img.convert('L')
        
        # Example: Detect health bar by color
        # This depends heavily on your specific game
        
        state = {}
        # Add your analysis logic here
        
        return state


    def send_key(self, key):
        """Send keyboard input to game"""
        print(f">>> Bot pressing: {key}")
        pyautogui.press(key)

    def start_game(self):
        self.send_key('n')
        time.sleep(25)
        self.send_key('G')
        time.sleep(2)
        self.send_key('enter')
        time.sleep(2)
        self.send_key('enter')

    def try_enter(self):
        time.sleep(5)
        self.send_key('enter')

    def try_f(self):
        time.sleep(5)
        self.send_key('f1')
        self.send_key('f3')
        self.send_key('f6')
        self.send_key('f8')
        

    def try_n(self):
        time.sleep(5)
        self.send_key('n')


    def try_esc(self):
        time.sleep(5)
        self.send_key('esc')



    def play(self):
        self.start_game()
        counter = 0
        while True:
            self.try_enter()
            screen = self.capture_screen()
            screen = Image.fromarray(screen)
            screen.save("Data\Screenshots\{}_enter.jpeg".format(counter))

            self.try_f()
            screen = self.capture_screen()
            screen = Image.fromarray(screen)
            screen.save("Data\Screenshots\{}_f.jpeg".format(counter))

            self.try_esc()
            self.try_esc()
            screen = self.capture_screen()
            screen = Image.fromarray(screen)
            screen.save("Data\Screenshots\{}_esc_esc.jpeg".format(counter))

            self.try_n()
            screen = self.capture_screen()
            screen = Image.fromarray(screen)
            screen.save("Data\Screenshots\{}_no.jpeg".format(counter))

            counter += 1
#Simulating decisions and saving the images each step

In [3]:
img_dir = os.listdir(r'Data\Screenshots')
last_img = None
for img_name in img_dir:
    img_path = os.path.join(r'Data\Screenshots', img_name)
    img = Image.open(img_path)
    if last_img is None:
        last_img = img
    elif ssim(np.asarray(last_img), np.asarray(img), channel_axis=2) >= 0.95:
        os.remove(img_path)
    else:
        last_img = img
#Delete duplicated images obtained from simulator

In [2]:
frames_folder = 'Data/Screenshots'

for frame in os.listdir(frames_folder):
    frame_path = os.path.join(frames_folder, frame)
    img = Image.open(frame_path)
    img.resize((200, 200)).show(title = frame_path)
    class_choice = input('Frame_class')
    if class_choice == 'sk':
        continue
    copy_frame_to_folder(frame_path, class_choice)
    img.close()